# Transformada de Fourier de Curto Prazo (STFT)

## 1. Informação Temporal

In [ ]:
import numpy as np
import librosa

import matplotlib.pyplot as plt

import IPython.display as ipd

In [ ]:
fs = 2000

duration = 0.1
f1 = 100
f2 = 200

N = int(duration * fs)
t = np.arange(N) / fs

t1 = t[:N//2]
t2 = t[N//2:]

x1 = 1.0 * np.sin(2 * np.pi * f1 * t1)
x2 = 1.0 * np.sin(2 * np.pi * f2 * t2)

x = np.concatenate((x1, x2))

In [ ]:
# Create figure with specific size (width=10, height=4)
plt.figure(figsize=(10, 4))

# Plot the data
plt.plot(t, x)

plt.xlabel('Tempo [s]')
plt.ylabel('Amplitude')

plt.tight_layout()
plt.show()

In [ ]:
X = np.abs(np.fft.fft(x))
freq = np.linspace(0, fs/2, N//2+1)

X = X[:N//2 + 1]

freq = freq[:N//2 + 1]

In [ ]:
plt.figure(figsize=(10, 4))

plt.stem(freq, X)

plt.xlabel('Frequência [Hz]')
plt.ylabel('Amplitude')

plt.tight_layout()
plt.show()

Applying the DFT over a long window does not reveal transitions in
spectral content

<div align="center">
  <img src="img/stft.png" alt="title" width="640">
</div>

## 2. STFT em Arquivos de Áudio

In [ ]:
# She had your dark suit in greasy wash water all year.
data_wav, sr_wav = librosa.load('wav\\SA1.wav', sr = 16000)

In [ ]:
data_wav.shape

In [ ]:
sr_wav

In [ ]:
plt.figure(figsize=(10, 4))

librosa.display.waveshow(data_wav,sr=sr_wav)

plt.xlabel('Tempo [s]')
plt.ylabel('Amplitude')


plt.tight_layout()
plt.show()

In [ ]:
ipd.Audio(data_wav, rate = sr_wav)

In [ ]:
hop_length = 160 # 160/16000 = 10 ms
n_window = 320 # 320/16000 = 20 ms

data_wav_pad = np.pad(data_wav, (n_window//2, n_window//2), 'constant', constant_values=(0,0))

slices = librosa.util.frame(data_wav_pad, frame_length = n_window, hop_length = hop_length)

In [ ]:
print(f'Audio shape: {data_wav.shape}')

In [ ]:
print(f'Sliced audio shape: {slices.shape}')

In [ ]:
slices.shape

In [ ]:
win = librosa.filters.get_window('hamming', n_window)

In [ ]:
win.shape

In [ ]:
win = win.reshape(n_window,1)

In [ ]:
win.shape

In [ ]:
slices_windowed = slices * win

In [ ]:
slices_windowed.shape

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 8))

ax[0].plot(slices[:, 251])
ax[0].set_title('Sinal Original')
ax[0].set_ylabel('Amplitude')
ax[0].set_xlim([0, len(slices[:, 251])])

ax[1].plot(slices_windowed[:, 251])
ax[1].set_title('Sinal Janelado')
ax[1].set_xlabel('Amostras')
ax[1].set_ylabel('Amplitude')
ax[1].set_xlim([0, len(slices_windowed[:, 251])])

plt.tight_layout()
plt.show()

Para cada segmento, calcular a DFT, e selecionar apenas a parte DC, as frequências positivas e a frequência de Nyquist.

In [ ]:
fft_slices = np.fft.fft(slices_windowed, axis=0)[0:n_window // 2 + 1]

Para gerar o espectrograma (exibição gráfica da magnitude da STFT discreta), podemos usar a magnitude ou a magnitude quadrada dos coeficientes da STFT, geralmente em escala logarítmica (dB).

In [ ]:
S = np.abs(fft_slices)
S = librosa.amplitude_to_db(S, ref = np.max(S), top_db = 100)
S

In [ ]:
S = np.abs(fft_slices)
S = 20 * np.log10(S/np.max(S)) 
S

In [ ]:
plt.figure(figsize=(11, 4)) 
librosa.display.specshow(S, y_axis = 'linear', x_axis = 'time', sr = sr_wav,
                         hop_length = hop_length, cmap='gray_r', vmin=-60) # verificar vmin=-60 and vmax=0
plt.colorbar(format='%+2.0f dB')

plt.tight_layout()
plt.show()

## 3.Librosa STFT

A função `librosa.stft` é utilizada para calcular a STFT (Short-Time Fourier Transform) 
de um sinal de áudio. Esta função recebe como principais argumentos:

| Parâmetro | Descrição |
|-----------|-----------|
| `n_fft` | Número de pontos da FFT |
| `hop_length` | Número de amostras entre janelas consecutivas |
| `win_length` | Tamanho da janela (frame size) |

In [ ]:
hop_length = 160
n_window = 320
n_fft = 320

X = librosa.stft(data_wav, win_length = n_window, n_fft = n_fft, hop_length = hop_length, 
                 window = 'hamming') # , center=False

S_librosa = librosa.amplitude_to_db(np.abs(X),ref=np.max, top_db = 100)

In [ ]:
plt.figure(figsize=(11, 4)) 
librosa.display.specshow(S_librosa, y_axis = 'linear', x_axis = 'time', sr = sr_wav, 
                         hop_length = hop_length, cmap='gray_r', vmin=-60)
plt.colorbar(format='%+2.0f dB')

plt.tight_layout()
plt.show()

Verificar : Centralização e padding: https://librosa.org/doc/main/generated/librosa.stft.html

Parâmetro `center` (centralizar ou não)

Parâmetro `pad_mode` (modo de preenchimento)

Usado **apenas quando `center=True`**. Define como as bordas do sinal são estendidas (padded). O valor é passado para `np.pad`.

| Modo (`pad_mode`) | Descrição |
| :--- | :--- |
| `'constant'` (padrão) | Preenche as bordas com um valor constante (padrão é zero). |
| `'edge'` | Preenche com o valor da borda (repetindo o primeiro/último valor). |
| `'linear_ramp'` | Preenche com uma rampa linear entre a borda e um valor final. |
| `'reflect'` | Reflete os valores das bordas, sem repetir o valor da borda. |
| `'symmetric'` | Reflete os valores das bordas, repetindo o valor da borda. |

## 4. Resolução Tempo $\times$ Frequência

In [ ]:
n_window = 80 # 80/16000 = 5 ms
hop_length = 20 # 1.25 ms
n_fft = 1024

X = librosa.stft(data_wav, win_length = n_window, n_fft = n_fft, hop_length = hop_length, 
                 window = 'hamming')

In [ ]:
plt.figure(figsize=(11, 4)) 

librosa.display.specshow(librosa.amplitude_to_db(np.abs(X), ref=np.max), 
                         y_axis = 'linear', x_axis = 'time', sr = 16000, 
                         hop_length = hop_length, cmap='gray_r', vmin=-50, vmax=0)
plt.colorbar(format='%+2.0f dB')

plt.tight_layout()
plt.show()

In [ ]:
hop_length = 240 # 15 ms
n_window = 480 # 30 ms
n_fft = 1024

X = librosa.stft(data_wav, win_length = n_window, n_fft = n_fft, hop_length = hop_length, 
                 window = 'hamming')

In [ ]:
plt.figure(figsize=(11, 4)) 

librosa.display.specshow(librosa.amplitude_to_db(np.abs(X),ref=np.max), 
                         y_axis = 'linear', x_axis = 'time', sr = 16000, 
                         hop_length = hop_length, cmap='gray_r', vmin=-60, vmax=0)
plt.colorbar(format='%+2.0f dB')

plt.tight_layout()
plt.show()

Janelas mais longas apresentam melhor resolução em frequência, à custa de uma redução na resolução temporal.

Janelas mais curtas fornecem uma representação mais detalhada no domínio do tempo, mas, em geral, resultam em baixa resolução em frequência.

Em resumo:

- Para uma janela longa, o resultado é o espectrograma de faixa estreita (*narrowband*), que evidencia a estrutura harmônica na forma de "estriações" horizontais

- Para uma janela curta, o resultado é o espectrograma de faixa larga (*wideband*), que evidencia a estrutura temporal periódica na forma de "estriações" verticais

<div align="center">
  <img src="img/nar.png" alt="title" width="800">
</div>

## 5. Variações do Espectrograma

A função `librosa.display.specshow` permite controlar a escala do eixo de frequência através do parâmetro `y_axis` (ou `x_axis` quando aplicável). Abaixo estão os principais tipos disponíveis e suas características:

| Tipo (`y_axis`) | Descrição | Aplicação Típica |
| :--- | :--- | :--- |
| **`'linear'`** | Escala linear de frequências.  | Sinais de banda estreita, componentes harmônicas simples. |
| **`'hz'`** | Sinônimo de `'linear'`. Escala linear em Hertz. | Mesmo que `'linear'`. |
| **`'log'`** | Escala logarítmica de frequências. | Sinais musicais e de voz. |
| **`'mel'`** | Escala mel. Frequências são mapeadas para a escala mel. | Análise de voz e música, extração de características para aprendizado de máquina. |
| **`'cqt_hz'`** | Frequências determinadas pela Transformada Q-Constante (CQT), exibidas em Hertz. A CQT possui resolução logarítmica natural. | Análise harmônica de música polifônica, onde intervalos musicais são preservados. |
| **`'cqt_note'`** | Similar ao `'cqt_hz'`, mas o eixo é rotulado em notas musicais (ex: C4, D4, ...). | Visualização intuitiva para análise musical, especialmente útil para músicos. |

In [ ]:
x, sr = librosa.load('wav\\handel.wav', sr = 8192)

In [ ]:
ipd.Audio(x, rate = sr)

In [ ]:
hop_length = 128
n_fft = 256
n_window = 256

X = librosa.stft(x, win_length = n_window, n_fft = n_fft, hop_length = hop_length, window = 'hamming')
D = librosa.amplitude_to_db(np.abs(X), ref=np.max)

plt.figure(figsize=(11, 4)) 
librosa.display.specshow(D, y_axis = 'linear', x_axis='time', sr = 8192, 
                         hop_length=hop_length, vmin=-50, vmax=0, cmap='gray_r') # 
plt.colorbar(format='%+2.0f dB')
plt.title('Linear-frequency power spectrogram')

plt.tight_layout()
plt.show()

### Escala logarítmica 

In [ ]:
plt.figure(figsize=(11, 4)) 
librosa.display.specshow(D, y_axis = 'log', x_axis='time', sr = 8192, 
                         hop_length=hop_length, vmin=-50, vmax=0)
plt.colorbar(format='%+2.0f dB')
plt.title('Log-frequency power spectrogram')

plt.tight_layout()
plt.show()

### Cromagrama
Um chroma vector é um vetor de 12 elementos, onde cada posição corresponde a uma das 12 classes de notas musicais da escala cromática.

In [ ]:
# https://www.onlinepianist.com/virtual-piano

chromagram = librosa.feature.chroma_stft(y = x, sr = sr, n_fft = 256, hop_length = 128, win_length=256)
plt.figure(figsize=(12, 4)) 
librosa.display.specshow(chromagram, y_axis = 'chroma', x_axis='time', 
                         sr = 8192, hop_length=hop_length, cmap = 'coolwarm')

plt.tight_layout()
plt.show()

### Espectrograma na escala Mel

In [ ]:
plt.figure(figsize=(11, 4)) 
librosa.display.specshow(D, y_axis = 'mel', x_axis='time', sr = 8192, 
                         hop_length=hop_length)
plt.colorbar(format='%+2.0f dB')
plt.title('Mel-frequency power spectrogram')
plt.tight_layout()
plt.show()

In [ ]:
hop_length = 512
n_fft = 1024
n_window = 1024

X = librosa.stft(x, win_length = n_window, n_fft = n_fft, hop_length = hop_length, window = 'hamming')
D = librosa.amplitude_to_db(np.abs(X), ref=np.max)

plt.figure(figsize=(11, 4)) 
librosa.display.specshow(D, y_axis = 'linear', x_axis='time', sr = sr, 
                         hop_length=hop_length) # , cmap='gray_r'
plt.colorbar(format='%+2.0f dB')
plt.title('Linear-frequency power spectrogram')

plt.tight_layout()
plt.show()

## 6. Exemplos de aplicações

### On Aphex Twin’s Windowlicker album, track 2, ∼ 5:27–5:37

![title](img/band.png)

Aphex Face

### Algoritmo do Shazam

Fingerprinting $\rightarrow$ Detecção de picos e Hashing

O Shazam cria pares utilizando o seguinte algoritmo:

- Seleciona um ponto (chamado de ponto âncora)
- Define uma zona alvo no espectrograma a partir desse ponto
- Para cada ponto dentro da zona alvo, cria um par com o ponto âncora

As três primeiras componentes ($f_A$, $f_B$ e $\Delta T$) formam o hash.  
As demais informações são utilizadas para localizar o hash em um instante específico da música, sendo usadas posteriormente na etapa de correspondência.

![title](img/shazam.png)

## Cadastro de uma música (registro)

O Shazam realiza os seguintes passos:

- Calcula o espectrograma da música
- Extrai os picos espectrais (regiões de maior energia)
- Combina esses picos em pares (gerando hashes)
- Armazena o conjunto de hashes como uma “impressão digital” (*fingerprint*) da música

## Reconhecimento de um áudio

Para identificar uma amostra de áudio, o Shazam:

- Calcula o fingerprint da amostra
- Busca hashes correspondentes no banco de dados
- Para cada música candidata:
  - Calcula a diferença entre o tempo da música e o tempo da amostra para cada hash correspondente
  - Agrupa esses valores em um histograma
  - Seleciona o maior pico do histograma como pontuação da música
- Retorna a música com maior pontuação

http://www.ee.columbia.edu/~dpwe/papers/Wang03-shazam.pdf

## Referências

MÜLLER, Meinard. *Fundamentals of Music Processing: Audio, Analysis, Algorithms, Applications*. Cham: Springer, 2015.

MÜLLER, Meinard. *STFT – Basic (FMP Notebooks)*. Erlangen: AudioLabs, Friedrich-Alexander-Universität Erlangen-Nürnberg, [2015]. Disponível em: <https://www.audiolabs-erlangen.de/resources/MIR/FMP/C2/C2_STFT-Basic.html>. Acesso em: 24 mar. 2026.

GUTIERREZ-OSUNA, Ricardo. *Introduction to Speech Processing*. College Station: Texas A\&M University (CSE@TAMU), [s.d.].

